In [368]:
import sim          
import sympy as sp  
import numpy as np
import time
import math

def connect(port):
    sim.simxFinish(-1)
    clientID=sim.simxStart('127.0.0.1',port,True,True,2000,5) # Conectarse
    if clientID == 0: print("conectado a", port)
    else: print("no se pudo conectar")
    return clientID

In [369]:
clientID = connect(19999)

retCode,sensorHandle=sim.simxGetObjectHandle(clientID,'Vision_sensor',sim.simx_opmode_blocking)
retCode, resolution, image=sim.simxGetVisionSensorImage(clientID,sensorHandle,0,sim.simx_opmode_oneshot_wait)

retCode,ruedaDerecha=sim.simxGetObjectHandle(clientID,'RuedaR',sim.simx_opmode_blocking)
retCode,ruedaIzquierda=sim.simxGetObjectHandle(clientID,'RuedaL',sim.simx_opmode_blocking)

retCode,suction=sim.simxGetObjectHandle(clientID,'suctionPad',sim.simx_opmode_blocking)

ret,ultrasonidoDerecha=sim.simxGetObjectHandle(clientID,'SensorR',sim.simx_opmode_blocking)
ret,ultrasonidoIzquierda=sim.simxGetObjectHandle(clientID,'SensorL',sim.simx_opmode_blocking)
ret,ultrasonidoDelante=sim.simxGetObjectHandle(clientID,'SensorD',sim.simx_opmode_blocking)
ret,ultrasonidoAtras=sim.simxGetObjectHandle(clientID,'SensorA',sim.simx_opmode_blocking)
ret,cuerpo=sim.simxGetObjectHandle(clientID,'AWSD',sim.simx_opmode_blocking)

conectado a 19999


In [370]:
def setEffector(val):
# function that triggers the end effector remotely
# val is Int with value 0 or 1 to disable or activate the final actuator.
    res,retInts,retFloats,retStrings,retBuffer=sim.simxCallScriptFunction(clientID,
        "suctionPad", sim.sim_scripttype_childscript,"setEffector",[val],[],[],"", sim.simx_opmode_blocking)
    return res

def obtenerDistanciaSensor(ultrasonido):
    errorCode, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector=sim.simxReadProximitySensor(clientID,ultrasonido, sim.simx_opmode_blocking)
    sensor_val=np.linalg.norm(detectedPoint)

    return sensor_val

In [ ]:
def moverCasilla(v, distancia=0.24):
    """
    v = velocidad angular de las ruedas (rad/s)
    distancia = 0.24 m
    """
    # Obtener posicion inicial (eje X o Y segun orientacion)
    # Usamos streaming para inicializar
    ret, pos_inicial = sim.simxGetObjectPosition(clientID, cuerpo, -1, sim.simx_opmode_blocking)
    if ret != 0:
        print("Error al obtener posición inicial")
        return
    inicio_x = pos_inicial[0]   # Suponiendo que avanza en X
    inicio_y = pos_inicial[1]   # Suponiendo que avanza en Y
    
    # Arrancar motores (modo streaming)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_streaming)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   v, sim.simx_opmode_streaming)
    
    # Activar streaming de la posicion porque si no, no puede ir leyendo la posicion actual
    sim.simxGetObjectPosition(clientID, cuerpo, -1, sim.simx_opmode_streaming)
    
    # Bucle de control
    while True:
        ret, pos_actual = sim.simxGetObjectPosition(clientID, cuerpo, -1, sim.simx_opmode_buffer)
        print(pos_actual)
        if ret == 0:   # Datos disponibles
            avanzado_x = pos_actual[0] - inicio_x
            avanzado_y = pos_actual[1] - inicio_y
            print(f"Avanzado: {avanzado_y:.3f} m")
            if abs(avanzado_x) >= distancia or abs(avanzado_y) >= distancia:
                break
        # Pequeña pausa para no saturar
        time.sleep(0.05)   # O time.sleep(0.05)
    
    # Detener ruedas, pero como la rueda izquierda para antes, genera un pequeño desvio
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, 0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_blocking)

     # Esperar un instante a que se disipen las inercias
    time.sleep(0.1)
    
    # Leer orientacion y si hay desviacion > tolerancia, aplicar micro-corrección
    ret, orientacion = sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_blocking)
    angulo_z = orientacion[2]
    if abs(angulo_z) > 0.001:
        # Girar en sentido contrario muy suavemente
        sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, 0.75, sim.simx_opmode_streaming)
        sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,  -0.75, sim.simx_opmode_streaming)
        time.sleep(0.05)
        sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, 0, sim.simx_opmode_blocking)
        sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_blocking)
    
    print("Movimiento completado")
    
def movimientoContinuo(velocidad, direccion):
    v = velocidad * direccion
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   v, sim.simx_opmode_blocking)
    

def giro90(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    tiempo = radianes_rueda / v
    print(tiempo)
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,  -v, sim.simx_opmode_blocking)
    
    time.sleep(tiempo)
    
    # 5. Frenamos ambos motores
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,    0, sim.simx_opmode_blocking)

def girar(angulo_grados, vel_rad_s, tolerancia=0.5):
    """
    Gira el robot exactamente 'angulo_grados' (positivo = derecha, negativo = izquierda)
    usando control de lazo cerrado.
    """
    # Obtener orientacion inicial
    ret, orientacion = sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_blocking)
    if ret != 0:
        print("Error al leer orientación inicial")
        return
    angulo_inicial = orientacion[2]
    
    # Determinar sentido de giro y velocidad de las ruedas
    if angulo_grados > 0:
        # Giro horario
        vel_izq =  vel_rad_s
        vel_der = -vel_rad_s
    else:
        # Giro antihorario
        vel_izq = -vel_rad_s
        vel_der =  vel_rad_s
        
    angulo_objetivo_rad = angulo_inicial - math.radians(angulo_grados)
    
    # Normalizar objetivo al rango [-π, π]
    angulo_objetivo_rad = math.atan2(math.sin(angulo_objetivo_rad), math.cos(angulo_objetivo_rad))
    
    # Iniciar movimiento
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, vel_izq, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha, vel_der, sim.simx_opmode_blocking)
    
    # Configurar streaming de orientación
    sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_streaming)
    
    tolerancia_rad = math.radians(tolerancia)
    while True:
        ret, orientacion = sim.simxGetObjectOrientation(clientID, cuerpo, -1, sim.simx_opmode_buffer)
        if ret == 0:
            angulo_actual = orientacion[2]
            # Diferencia angular minima (evita problemas con el cruce de ±π)
            error = angulo_objetivo_rad - angulo_actual
            error = math.atan2(math.sin(error), math.cos(error))
            if abs(error) < tolerancia_rad:
                break
        time.sleep(0.05)
    
    # Detener el robot
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda, 0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha, 0, sim.simx_opmode_blocking)
    
def giroContinuo(v, direccion):
    radianes_rueda = math.radians(90 * direccion) * 0.17 / 0.08
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  v, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   -v, sim.simx_opmode_blocking)

def detener():
    sim.simxSetJointTargetVelocity(clientID, ruedaIzquierda,  0, sim.simx_opmode_blocking)
    sim.simxSetJointTargetVelocity(clientID, ruedaDerecha,   0, sim.simx_opmode_blocking)
                


In [372]:
detener()
derecha = obtenerDistanciaSensor(ultrasonidoDerecha)
izquierda = obtenerDistanciaSensor(ultrasonidoIzquierda)
delante = obtenerDistanciaSensor(ultrasonidoDelante)
atras = obtenerDistanciaSensor(ultrasonidoAtras)

print(derecha)
print(izquierda)

0.054998603053636214
0.05500232091445758


In [373]:
#moverCasilla(3)
moverCasilla_lazo_cerrado(3)
#girar_exacto(-90, 0.8)

[0.0, 0.0, 0.0]
[0.0, 0.0, 0.0]
[-1.3532304592445143e-06, 0.0004901540814898908, 0.09500116854906082]
Avanzado: 0.000 m
[0.0002944484003819525, 1.8229127363156294e-06, 0.09500075876712799]
Avanzado: -0.000 m
[0.0005973638617433608, -0.0016158890211954713, 0.09499993175268173]
Avanzado: -0.002 m
[0.0005973638617433608, -0.0016158890211954713, 0.09499993175268173]
Avanzado: -0.002 m
[0.0007787811337038875, -0.004460207186639309, 0.09499823302030563]
Avanzado: -0.005 m
[0.0008713300921954215, -0.008357993327081203, 0.09500470012426376]
Avanzado: -0.009 m
[0.0008713300921954215, -0.008357993327081203, 0.09500470012426376]
Avanzado: -0.009 m
[0.0009724256815388799, -0.013398071750998497, 0.0950111448764801]
Avanzado: -0.014 m
[0.00106237328145653, -0.01929689198732376, 0.09501499682664871]
Avanzado: -0.020 m
[0.0011276007862761617, -0.025263097137212753, 0.09501629322767258]
Avanzado: -0.026 m
[0.0011276007862761617, -0.025263097137212753, 0.09501629322767258]
Avanzado: -0.026 m
[0.00120628

KeyboardInterrupt: 